In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/Final_Dataset/Sam_final_dataset/Audio/voice_all_files/FINAL_SYNC_MASTER.csv")

print("COLUMNS:")
print(df.columns)

print("\nHEAD:")
print(df.head())

print("\nUNIQUE attack_type (if exists):")
if "attack_type" in df.columns:
    print(df["attack_type"].unique())

print("\nUNIQUE label (if exists):")
if "label" in df.columns:
    print(df["label"].unique())

COLUMNS:
Index(['user_id', 'session_token', 'video_filename', 'video_quality',
       'full_path', 'session_token_clean', 'audio_quality'],
      dtype='object')

HEAD:
                                user_id              session_token  \
0  28f503ad-299f-4a2f-96c7-ab22875e0e18  1769877865550-cngi8444j34   
1  f06f8bb5-8303-416b-bf9c-e247eb17b864  1770008423822-5ek366c1tyi   
2  4cf15e9c-2f70-47cb-803a-52bb527eb74d  1770364827761-mr9ug9dhlib   
3  a5ad9b9a-aadb-40b9-8d74-7baf4bb2fb9f  1770016503680-7cdim9wd80u   
4  585fb1cb-bb40-44ae-85c5-498c971c3436  1770206737552-7b84if03p6b   

                                      video_filename   video_quality  \
0             1769877865550-cngi8444j34-video 8.webm      With Specs   
1            1770008423822-5ek366c1tyi-video 11.webm  perfect videos   
2                                                NaN        no_video   
3               1770016503680-7cdim9wd80u-video.webm  perfect videos   
4  madhumitha-balajeyandhan-1770206737552-7b84if0.

unzip tts files

In [6]:
import os

base_audio = "/content/drive/MyDrive/Final_Dataset/Sam_final_dataset/Audio/voice_all_files"

os.makedirs(os.path.join(base_audio, "tts_attacks_final"), exist_ok=True)
os.makedirs(os.path.join(base_audio, "tts_eval_file"), exist_ok=True)

!unzip -o "/content/drive/MyDrive/Final_Dataset/Sam_final_dataset/Audio/voice_all_files/tts_attacks_final.zip" -d "/content/drive/MyDrive/Final_Dataset/Sam_final_dataset/Audio/voice_all_files/tts_attacks_final"

!unzip -o "/content/drive/MyDrive/Final_Dataset/Sam_final_dataset/Audio/voice_all_files/tts_eval_file.zip" -d "/content/drive/MyDrive/Final_Dataset/Sam_final_dataset/Audio/voice_all_files/tts_eval_file"

Archive:  /content/drive/MyDrive/Final_Dataset/Sam_final_dataset/Audio/voice_all_files/tts_attacks_final.zip
  inflating: /content/drive/MyDrive/Final_Dataset/Sam_final_dataset/Audio/voice_all_files/tts_attacks_final/attack_1769877865550-cngi8444j34.wav  
  inflating: /content/drive/MyDrive/Final_Dataset/Sam_final_dataset/Audio/voice_all_files/tts_attacks_final/attack_1770008423822-5ek366c1tyi.wav  
  inflating: /content/drive/MyDrive/Final_Dataset/Sam_final_dataset/Audio/voice_all_files/tts_attacks_final/attack_1770016503680-7cdim9wd80u.wav  
  inflating: /content/drive/MyDrive/Final_Dataset/Sam_final_dataset/Audio/voice_all_files/tts_attacks_final/attack_1770206737552-7b84if03p6b.wav  
  inflating: /content/drive/MyDrive/Final_Dataset/Sam_final_dataset/Audio/voice_all_files/tts_attacks_final/attack_1770188000031-tgsearqo8in.wav  
  inflating: /content/drive/MyDrive/Final_Dataset/Sam_final_dataset/Audio/voice_all_files/tts_attacks_final/attack_1770088530588-5efqabvflf3.wav  
  inflati

verify where the .wav files actually are

In [7]:
import os

paths_to_check = [
    "/content/drive/MyDrive/Final_Dataset/Sam_final_dataset/Audio/replay_attacks_wav",
    "/content/drive/MyDrive/Final_Dataset/Sam_final_dataset/Audio/replay_attacks_m4a_and_wav",
    "/content/drive/MyDrive/Final_Dataset/Sam_final_dataset/Audio/voice_all_files",
    "/content/drive/MyDrive/Final_Dataset/Sam_final_dataset/Audio/voice_all_files/tts_attacks_final",
    "/content/drive/MyDrive/Final_Dataset/Sam_final_dataset/Audio/voice_all_files/tts_eval_file",
]

for p in paths_to_check:
    wav_count = 0
    for root, _, files in os.walk(p):
        wav_count += sum(f.lower().endswith(".wav") for f in files)
    print(p, "->", wav_count, ".wav files")

/content/drive/MyDrive/Final_Dataset/Sam_final_dataset/Audio/replay_attacks_wav -> 118 .wav files
/content/drive/MyDrive/Final_Dataset/Sam_final_dataset/Audio/replay_attacks_m4a_and_wav -> 118 .wav files
/content/drive/MyDrive/Final_Dataset/Sam_final_dataset/Audio/voice_all_files -> 284 .wav files
/content/drive/MyDrive/Final_Dataset/Sam_final_dataset/Audio/voice_all_files/tts_attacks_final -> 118 .wav files
/content/drive/MyDrive/Final_Dataset/Sam_final_dataset/Audio/voice_all_files/tts_eval_file -> 166 .wav files


use the correct metadata builder

In [8]:
import os
import pandas as pd

base_audio = "/content/drive/MyDrive/Final_Dataset/Sam_final_dataset/Audio"

genuine_dir = os.path.join(base_audio, "voice_all_files")
replay_dir_1 = os.path.join(base_audio, "replay_attacks_wav")
replay_dir_2 = os.path.join(base_audio, "replay_attacks_m4a_and_wav")
tts_dir_1 = os.path.join(base_audio, "voice_all_files", "tts_attacks_final")
tts_dir_2 = os.path.join(base_audio, "voice_all_files", "tts_eval_file")

rows = []

def collect_audio(folder, label, attack_type, exts=(".wav",)):
    if not os.path.exists(folder):
        print("Missing folder:", folder)
        return

    for root, _, files in os.walk(folder):
        for f in files:
            if f.lower().endswith(exts):
                rows.append({
                    "file_path": os.path.join(root, f),
                    "label": label,
                    "attack_type": attack_type
                })

# Genuine
collect_audio(genuine_dir, label=0, attack_type="genuine", exts=(".wav",))

# Replay
collect_audio(replay_dir_1, label=1, attack_type="replay", exts=(".wav",))
collect_audio(replay_dir_2, label=1, attack_type="replay", exts=(".wav", ".m4a"))

# TTS
collect_audio(tts_dir_1, label=1, attack_type="tts", exts=(".wav",))
collect_audio(tts_dir_2, label=1, attack_type="tts", exts=(".wav",))

voice_df = pd.DataFrame(rows)

print("Dataset size:", voice_df.shape)
if not voice_df.empty:
    print(voice_df["attack_type"].value_counts())
    print(voice_df.head())
else:
    print("No audio files found.")

Dataset size: (922, 3)
attack_type
replay     354
genuine    284
tts        284
Name: count, dtype: int64
                                           file_path  label attack_type
0  /content/drive/MyDrive/Final_Dataset/Sam_final...      0     genuine
1  /content/drive/MyDrive/Final_Dataset/Sam_final...      0     genuine
2  /content/drive/MyDrive/Final_Dataset/Sam_final...      0     genuine
3  /content/drive/MyDrive/Final_Dataset/Sam_final...      0     genuine
4  /content/drive/MyDrive/Final_Dataset/Sam_final...      0     genuine


check filename duplicates

In [9]:
import os
import pandas as pd

attack_dirs = [
    "/content/drive/MyDrive/Final_Dataset/Sam_final_dataset/Audio/replay_attacks_wav",
    "/content/drive/MyDrive/Final_Dataset/Sam_final_dataset/Audio/replay_attacks_m4a_and_wav",
    "/content/drive/MyDrive/Final_Dataset/Sam_final_dataset/Audio/voice_all_files/tts_attacks_final",
    "/content/drive/MyDrive/Final_Dataset/Sam_final_dataset/Audio/voice_all_files/tts_eval_file"
]

files = []

for folder in attack_dirs:
    for root, _, filenames in os.walk(folder):
        for f in filenames:
            if f.lower().endswith((".wav", ".m4a")):
                files.append({
                    "file_name": f,
                    "file_path": os.path.join(root, f)
                })

attack_df = pd.DataFrame(files)

print("Total attack files:", len(attack_df))
attack_df.head()

Total attack files: 638


,file_name,file_path
0,1770016503680-7cdim9wd80u-audio_RA.webm.wav,/content/drive/MyDrive/Final_Dataset/Sam_final...
1,1769879335590-h4i23wgvh1-audio_RA.webm.wav,/content/drive/MyDrive/Final_Dataset/Sam_final...
2,avinash-jeevaratnam-1770301049346-y55avo4p2vi-...,/content/drive/MyDrive/Final_Dataset/Sam_final...
3,abinanth-sivagnanam-1770213756748-gezlecdaxw6-...,/content/drive/MyDrive/Final_Dataset/Sam_final...
4,dharushana-manohar-1770626604878-066mir6831a-a...,/content/drive/MyDrive/Final_Dataset/Sam_final...


In [10]:
dup_names = attack_df["file_name"].duplicated().sum()
print("Duplicate filenames:", dup_names)

attack_df[attack_df["file_name"].duplicated(keep=False)].head()

Duplicate filenames: 118


,file_name,file_path
0,1770016503680-7cdim9wd80u-audio_RA.webm.wav,/content/drive/MyDrive/Final_Dataset/Sam_final...
1,1769879335590-h4i23wgvh1-audio_RA.webm.wav,/content/drive/MyDrive/Final_Dataset/Sam_final...
2,avinash-jeevaratnam-1770301049346-y55avo4p2vi-...,/content/drive/MyDrive/Final_Dataset/Sam_final...
3,abinanth-sivagnanam-1770213756748-gezlecdaxw6-...,/content/drive/MyDrive/Final_Dataset/Sam_final...
4,dharushana-manohar-1770626604878-066mir6831a-a...,/content/drive/MyDrive/Final_Dataset/Sam_final...


In [11]:
import hashlib

def get_file_hash(path):
    try:
        with open(path, "rb") as f:
            return hashlib.md5(f.read()).hexdigest()
    except:
        return None

attack_df["hash"] = attack_df["file_path"].apply(get_file_hash)

dup_hashes = attack_df["hash"].duplicated().sum()
print("Duplicate audio files (same content):", dup_hashes)

Duplicate audio files (same content): 118


In [12]:
clean_attack_df = attack_df.drop_duplicates(subset=["hash"]).reset_index(drop=True)

print("Before:", len(attack_df))
print("After:", len(clean_attack_df))
print("Removed duplicates:", len(attack_df) - len(clean_attack_df))

Before: 638
After: 520
Removed duplicates: 118


In [14]:
voice_df = clean_attack_df.copy()

def infer_attack_type(path):
    path = path.lower()
    if "voice_all_files" in path:
        return "genuine"
    elif "replay" in path:
        return "replay"
    elif "tts" in path:
        return "tts"
    else:
        return "unknown"

voice_df["attack_type"] = voice_df["file_path"].apply(infer_attack_type)
voice_df["label"] = voice_df["attack_type"].apply(lambda x: 0 if x == "genuine" else 1)

print("Shape:", voice_df.shape)
print("\nAttack counts:")
print(voice_df["attack_type"].value_counts())

print("\nLabel counts:")
print(voice_df["label"].value_counts())

voice_df.head()

Shape: (520, 5)

Attack counts:
attack_type
genuine    284
replay     236
Name: count, dtype: int64

Label counts:
label
0    284
1    236
Name: count, dtype: int64


,file_name,file_path,hash,attack_type,label
0,1770016503680-7cdim9wd80u-audio_RA.webm.wav,/content/drive/MyDrive/Final_Dataset/Sam_final...,0269a8b2449a157a40f2a413d2a6db43,replay,1
1,1769879335590-h4i23wgvh1-audio_RA.webm.wav,/content/drive/MyDrive/Final_Dataset/Sam_final...,4dbbbc12c57cc5f94c6b08f4a5a02951,replay,1
2,avinash-jeevaratnam-1770301049346-y55avo4p2vi-...,/content/drive/MyDrive/Final_Dataset/Sam_final...,95c0128895a1612bb8d9fbf64fd5f47d,replay,1
3,abinanth-sivagnanam-1770213756748-gezlecdaxw6-...,/content/drive/MyDrive/Final_Dataset/Sam_final...,e5e29bf039ccfaa2b06fe32fd52fdad1,replay,1
4,dharushana-manohar-1770626604878-066mir6831a-a...,/content/drive/MyDrive/Final_Dataset/Sam_final...,5199a879b196df22dd603a60ff38544c,replay,1


In [15]:
print(voice_df[voice_df["attack_type"] == "unknown"].head())
print("Unknown rows:", (voice_df["attack_type"] == "unknown").sum())

Empty DataFrame
Columns: [file_name, file_path, hash, attack_type, label]
Index: []
Unknown rows: 0


In [16]:
voice_df = voice_df.drop(columns=["hash"], errors="ignore")

save_path = "/content/drive/MyDrive/Final_Dataset/Sam_final_dataset/Audio/final_voice_dataset_clean.csv"
voice_df.to_csv(save_path, index=False)

print("Saved clean dataset:", save_path)

Saved clean dataset: /content/drive/MyDrive/Final_Dataset/Sam_final_dataset/Audio/final_voice_dataset_clean.csv


In [17]:
import os

paths_to_check = [
    "/content/drive/MyDrive/Final_Dataset/Sam_final_dataset/Audio/voice_all_files/tts_attacks_final",
    "/content/drive/MyDrive/Final_Dataset/Sam_final_dataset/Audio/voice_all_files/tts_eval_file",
]

for p in paths_to_check:
    wav_count = 0
    for root, _, files in os.walk(p):
        wav_count += sum(f.lower().endswith(".wav") for f in files)
    print(p, "->", wav_count, ".wav files")

/content/drive/MyDrive/Final_Dataset/Sam_final_dataset/Audio/voice_all_files/tts_attacks_final -> 118 .wav files
/content/drive/MyDrive/Final_Dataset/Sam_final_dataset/Audio/voice_all_files/tts_eval_file -> 166 .wav files


In [18]:
import os
import pandas as pd
import hashlib

base_audio = "/content/drive/MyDrive/Final_Dataset/Sam_final_dataset/Audio"

genuine_dir = os.path.join(base_audio, "voice_all_files")
replay_dir_1 = os.path.join(base_audio, "replay_attacks_wav")
replay_dir_2 = os.path.join(base_audio, "replay_attacks_m4a_and_wav")
tts_dir_1 = os.path.join(base_audio, "voice_all_files", "tts_attacks_final")
tts_dir_2 = os.path.join(base_audio, "voice_all_files", "tts_eval_file")

rows = []

def collect_audio(folder, attack_type, exts=(".wav",)):
    if not os.path.exists(folder):
        print("Missing folder:", folder)
        return

    for root, _, files in os.walk(folder):
        for f in files:
            if f.lower().endswith(exts):
                rows.append({
                    "file_name": f,
                    "file_path": os.path.join(root, f),
                    "attack_type": attack_type
                })

# Collect all
collect_audio(genuine_dir, "genuine", exts=(".wav",))
collect_audio(replay_dir_1, "replay", exts=(".wav",))
collect_audio(replay_dir_2, "replay", exts=(".wav", ".m4a"))
collect_audio(tts_dir_1, "tts", exts=(".wav",))
collect_audio(tts_dir_2, "tts", exts=(".wav",))

voice_df = pd.DataFrame(rows)

print("Before dedup:", voice_df.shape)
print(voice_df["attack_type"].value_counts())

def get_file_hash(path):
    try:
        with open(path, "rb") as f:
            return hashlib.md5(f.read()).hexdigest()
    except:
        return None

voice_df["hash"] = voice_df["file_path"].apply(get_file_hash)

# Remove exact duplicate audio content
voice_df = voice_df.drop_duplicates(subset=["hash"]).reset_index(drop=True)

# Rebuild label after dedup
voice_df["label"] = voice_df["attack_type"].apply(lambda x: 0 if x == "genuine" else 1)

print("\nAfter dedup:", voice_df.shape)
print(voice_df["attack_type"].value_counts())
print("\nLabel counts:")
print(voice_df["label"].value_counts())

print("\nUnknown/empty hashes:", voice_df["hash"].isna().sum())

Before dedup: (922, 3)
attack_type
replay     354
genuine    284
tts        284
Name: count, dtype: int64

After dedup: (520, 5)
attack_type
genuine    284
replay     236
Name: count, dtype: int64

Label counts:
label
0    284
1    236
Name: count, dtype: int64

Unknown/empty hashes: 0


In [19]:
voice_df = voice_df.drop(columns=["hash"], errors="ignore")

save_path = "/content/drive/MyDrive/Final_Dataset/Sam_final_dataset/Audio/final_voice_dataset_clean.csv"
voice_df.to_csv(save_path, index=False)

print("Saved:", save_path)

Saved: /content/drive/MyDrive/Final_Dataset/Sam_final_dataset/Audio/final_voice_dataset_clean.csv


In [20]:
import os

base_audio = "/content/drive/MyDrive/Final_Dataset/Sam_final_dataset/Audio/voice_all_files"
genuine_zip = os.path.join(base_audio, "files_used_for_replay_attack.zip")
genuine_out = os.path.join(base_audio, "genuine")

os.makedirs(genuine_out, exist_ok=True)

!unzip -o "{genuine_zip}" -d "{genuine_out}"

Archive:  /content/drive/MyDrive/Final_Dataset/Sam_final_dataset/Audio/voice_all_files/files_used_for_replay_attack.zip
   creating: /content/drive/MyDrive/Final_Dataset/Sam_final_dataset/Audio/voice_all_files/genuine/replay_recording_files/
  inflating: /content/drive/MyDrive/Final_Dataset/Sam_final_dataset/Audio/voice_all_files/genuine/replay_recording_files/1769582224434-2n71gdsz237-audio.wav  
  inflating: /content/drive/MyDrive/Final_Dataset/Sam_final_dataset/Audio/voice_all_files/genuine/replay_recording_files/1769656704608-ef07jxxzhzn-audio.wav  
  inflating: /content/drive/MyDrive/Final_Dataset/Sam_final_dataset/Audio/voice_all_files/genuine/replay_recording_files/1769661288326-qjv26yfs9b-audio.wav  
  inflating: /content/drive/MyDrive/Final_Dataset/Sam_final_dataset/Audio/voice_all_files/genuine/replay_recording_files/1769678217131-a1yyynu4zxe-audio.wav  
  inflating: /content/drive/MyDrive/Final_Dataset/Sam_final_dataset/Audio/voice_all_files/genuine/replay_recording_files/17

In [21]:
import os
import pandas as pd
import hashlib

base_audio = "/content/drive/MyDrive/Final_Dataset/Sam_final_dataset/Audio"

genuine_dir = os.path.join(base_audio, "voice_all_files", "genuine")
replay_dir_1 = os.path.join(base_audio, "replay_attacks_wav")
replay_dir_2 = os.path.join(base_audio, "replay_attacks_m4a_and_wav")
tts_dir_1 = os.path.join(base_audio, "voice_all_files", "tts_attacks_final")
tts_dir_2 = os.path.join(base_audio, "voice_all_files", "tts_eval_file")

rows = []

def collect_audio(folder, attack_type):
    if not os.path.exists(folder):
        print("Missing folder:", folder)
        return

    for root, _, files in os.walk(folder):
        for f in files:
            if f.lower().endswith(".wav"):  # ✅ ONLY WAV
                rows.append({
                    "file_name": f,
                    "file_path": os.path.join(root, f),
                    "attack_type": attack_type
                })

# Genuine
collect_audio(genuine_dir, "genuine")

# Replay
collect_audio(replay_dir_1, "replay")
collect_audio(replay_dir_2, "replay")

# TTS
collect_audio(tts_dir_1, "tts")
collect_audio(tts_dir_2, "tts")

voice_df = pd.DataFrame(rows)

print("Before dedup:", voice_df.shape)
print("\nAttack distribution:")
print(voice_df["attack_type"].value_counts())

Before dedup: (638, 3)

Attack distribution:
attack_type
tts        284
replay     236
genuine    118
Name: count, dtype: int64


In [22]:
def get_file_hash(path):
    try:
        with open(path, "rb") as f:
            return hashlib.md5(f.read()).hexdigest()
    except:
        return None

voice_df["hash"] = voice_df["file_path"].apply(get_file_hash)

voice_df = voice_df.drop_duplicates(subset=["hash"]).reset_index(drop=True)

voice_df["label"] = voice_df["attack_type"].apply(lambda x: 0 if x == "genuine" else 1)

print("\nAfter dedup:", voice_df.shape)
print("\nAttack distribution:")
print(voice_df["attack_type"].value_counts())

print("\nLabel distribution:")
print(voice_df["label"].value_counts())


After dedup: (402, 5)

Attack distribution:
attack_type
tts        166
genuine    118
replay     118
Name: count, dtype: int64

Label distribution:
label
1    284
0    118
Name: count, dtype: int64


In [23]:
voice_df = voice_df.drop(columns=["hash"], errors="ignore")

save_path = "/content/drive/MyDrive/Final_Dataset/Sam_final_dataset/Audio/final_voice_dataset_clean_fixed.csv"
voice_df.to_csv(save_path, index=False)

print("Saved to:", save_path)

Saved to: /content/drive/MyDrive/Final_Dataset/Sam_final_dataset/Audio/final_voice_dataset_clean_fixed.csv


check the clean voice dataset

In [24]:
import os
import pandas as pd

voice_df = pd.read_csv("/content/drive/MyDrive/Final_Dataset/Sam_final_dataset/Audio/final_voice_dataset_clean_fixed.csv")

print("Shape:", voice_df.shape)
print("\nAttack distribution:")
print(voice_df["attack_type"].value_counts())
print("\nLabel distribution:")
print(voice_df["label"].value_counts())

missing = voice_df["file_path"].apply(lambda x: not os.path.exists(x)).sum()
print("\nMissing files:", missing)
print("\nHead:")
print(voice_df.head())

Shape: (402, 4)

Attack distribution:
attack_type
tts        166
genuine    118
replay     118
Name: count, dtype: int64

Label distribution:
label
1    284
0    118
Name: count, dtype: int64

Missing files: 0

Head:
                             file_name  \
0  1769582224434-2n71gdsz237-audio.wav   
1  1769656704608-ef07jxxzhzn-audio.wav   
2   1769661288326-qjv26yfs9b-audio.wav   
3  1769678217131-a1yyynu4zxe-audio.wav   
4  1769680362558-n9br5ma9t5p-audio.wav   

                                           file_path attack_type  label  
0  /content/drive/MyDrive/Final_Dataset/Sam_final...     genuine      0  
1  /content/drive/MyDrive/Final_Dataset/Sam_final...     genuine      0  
2  /content/drive/MyDrive/Final_Dataset/Sam_final...     genuine      0  
3  /content/drive/MyDrive/Final_Dataset/Sam_final...     genuine      0  
4  /content/drive/MyDrive/Final_Dataset/Sam_final...     genuine      0  


imports

In [27]:
import os
import gc
import numpy as np
import pandas as pd
import librosa

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix, accuracy_score

check device


In [28]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Using device: cuda
GPU: Tesla T4


load cleaned dataset

In [29]:
df = pd.read_csv("/content/drive/MyDrive/Final_Dataset/Sam_final_dataset/Audio/final_voice_dataset_clean_fixed.csv")

print("Shape:", df.shape)
print(df.head())
print("\nAttack distribution:")
print(df["attack_type"].value_counts())
print("\nLabel distribution:")
print(df["label"].value_counts())

Shape: (402, 4)
                             file_name  \
0  1769582224434-2n71gdsz237-audio.wav   
1  1769656704608-ef07jxxzhzn-audio.wav   
2   1769661288326-qjv26yfs9b-audio.wav   
3  1769678217131-a1yyynu4zxe-audio.wav   
4  1769680362558-n9br5ma9t5p-audio.wav   

                                           file_path attack_type  label  
0  /content/drive/MyDrive/Final_Dataset/Sam_final...     genuine      0  
1  /content/drive/MyDrive/Final_Dataset/Sam_final...     genuine      0  
2  /content/drive/MyDrive/Final_Dataset/Sam_final...     genuine      0  
3  /content/drive/MyDrive/Final_Dataset/Sam_final...     genuine      0  
4  /content/drive/MyDrive/Final_Dataset/Sam_final...     genuine      0  

Attack distribution:
attack_type
tts        166
genuine    118
replay     118
Name: count, dtype: int64

Label distribution:
label
1    284
0    118
Name: count, dtype: int64


keep only valid files

In [30]:
df = df.dropna(subset=["file_path", "label"]).copy()
df["label"] = df["label"].astype(int)

df["exists"] = df["file_path"].apply(os.path.exists)
print("Missing files:", (~df["exists"]).sum())

df = df[df["exists"]].drop(columns=["exists"]).reset_index(drop=True)
print("Final usable samples:", len(df))

Missing files: 0
Final usable samples: 402


train validation split

In [31]:
train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print("Train shape:", train_df.shape)
print("Val shape:", val_df.shape)
print("\nTrain labels:")
print(train_df["label"].value_counts())
print("\nVal labels:")
print(val_df["label"].value_counts())

Train shape: (321, 4)
Val shape: (81, 4)

Train labels:
label
1    227
0     94
Name: count, dtype: int64

Val labels:
label
1    57
0    24
Name: count, dtype: int64


audio feature extraction

In [32]:
def extract_logmel(
    file_path,
    sr=16000,
    duration=4.0,
    n_mels=64,
    n_fft=1024,
    hop_length=256
):
    y, _ = librosa.load(file_path, sr=sr)

    target_len = int(sr * duration)

    if len(y) < target_len:
        y = np.pad(y, (0, target_len - len(y)))
    else:
        y = y[:target_len]

    mel = librosa.feature.melspectrogram(
        y=y,
        sr=sr,
        n_fft=n_fft,
        hop_length=hop_length,
        n_mels=n_mels
    )

    logmel = librosa.power_to_db(mel, ref=np.max)
    return logmel.astype(np.float32)

dataset

In [33]:
class VoiceSpoofDataset(Dataset):
    def __init__(self, meta_df):
        self.meta_df = meta_df.reset_index(drop=True)

    def __len__(self):
        return len(self.meta_df)

    def __getitem__(self, idx):
        row = self.meta_df.iloc[idx]

        file_path = row["file_path"]
        label = int(row["label"])
        attack_type = row.get("attack_type", None)

        try:
            feat = extract_logmel(file_path)
        except Exception:
            feat = np.zeros((64, 251), dtype=np.float32)

        feat = (feat - feat.mean()) / (feat.std() + 1e-6)
        feat = torch.tensor(feat).unsqueeze(0)  # [1, mel, time]

        return feat, label, file_path, attack_type

loaders

In [34]:
gc.collect()
torch.cuda.empty_cache()

train_dataset = VoiceSpoofDataset(train_df)
val_dataset = VoiceSpoofDataset(val_df)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=0)

print("Train batches:", len(train_loader))
print("Val batches:", len(val_loader))

Train batches: 21
Val batches: 6


model

In [35]:
class SimpleAudioCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.pool = nn.AdaptiveAvgPool2d((1, 1))

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(32, 2)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        x = self.classifier(x)
        return x

model = SimpleAudioCNN().to(device)
print(model)

SimpleAudioCNN(
  (features): Sequential(
    (0): Conv2d(1, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (5): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (8): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU()
    (11): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (pool): AdaptiveAvgPool2d(output_size=(1, 1))
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=64, out_feature

loss and optimizer

In [36]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

Train / validate functions

In [37]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for feats, labels, _, _ in loader:
        feats = feats.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(feats)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * feats.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return running_loss / total, correct / total

def validate_one_epoch(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    all_labels = []
    all_preds = []
    all_probs = []

    with torch.no_grad():
        for feats, labels, _, _ in loader:
            feats = feats.to(device)
            labels = labels.to(device)

            outputs = model(feats)
            loss = criterion(outputs, labels)

            probs = torch.softmax(outputs, dim=1)[:, 1]
            preds = outputs.argmax(dim=1)

            running_loss += loss.item() * feats.size(0)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    auc = roc_auc_score(all_labels, all_probs)
    return running_loss / total, correct / total, auc, all_labels, all_preds, all_probs

train

In [38]:
num_epochs = 10
best_val_auc = 0.0
best_model_path = "/content/drive/MyDrive/Final_Dataset/Sam_final_dataset/Audio/voice_binary_cnn_best.pth"

for epoch in range(num_epochs):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc, val_auc, all_labels, all_preds, all_probs = validate_one_epoch(
        model, val_loader, criterion, device
    )

    print(f"\nEpoch {epoch+1}/{num_epochs}")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val   Loss: {val_loss:.4f} | Val   Acc: {val_acc:.4f} | Val AUC: {val_auc:.4f}")

    if val_auc > best_val_auc:
        best_val_auc = val_auc
        torch.save(model.state_dict(), best_model_path)
        print("Saved best model.")


Epoch 1/10
Train Loss: 0.5965 | Train Acc: 0.6760
Val   Loss: 0.5771 | Val   Acc: 0.6914 | Val AUC: 0.8640
Saved best model.

Epoch 2/10
Train Loss: 0.5037 | Train Acc: 0.7632
Val   Loss: 0.4787 | Val   Acc: 0.7037 | Val AUC: 0.8743
Saved best model.

Epoch 3/10
Train Loss: 0.4530 | Train Acc: 0.7788
Val   Loss: 0.4855 | Val   Acc: 0.8025 | Val AUC: 0.8801
Saved best model.

Epoch 4/10
Train Loss: 0.4327 | Train Acc: 0.8100
Val   Loss: 0.5429 | Val   Acc: 0.7778 | Val AUC: 0.8808
Saved best model.

Epoch 5/10
Train Loss: 0.4019 | Train Acc: 0.8318
Val   Loss: 0.6444 | Val   Acc: 0.7160 | Val AUC: 0.8918
Saved best model.

Epoch 6/10
Train Loss: 0.3744 | Train Acc: 0.8536
Val   Loss: 0.3783 | Val   Acc: 0.8148 | Val AUC: 0.9064
Saved best model.

Epoch 7/10
Train Loss: 0.3828 | Train Acc: 0.8442
Val   Loss: 0.4489 | Val   Acc: 0.8025 | Val AUC: 0.8955

Epoch 8/10
Train Loss: 0.3646 | Train Acc: 0.8442
Val   Loss: 0.4120 | Val   Acc: 0.8519 | Val AUC: 0.9020

Epoch 9/10
Train Loss: 0.34

load best model & evaluate

In [39]:
model.load_state_dict(torch.load(best_model_path, map_location=device))
model.eval()

val_loss, val_acc, val_auc, all_labels, all_preds, all_probs = validate_one_epoch(
    model, val_loader, criterion, device
)

print("\nVOICE RESULTS")
print("Loss:", val_loss)
print("Accuracy:", val_acc)
print("ROC-AUC:", val_auc)
print("\nClassification Report:")
print(classification_report(all_labels, all_preds))
print("\nConfusion Matrix:")
print(confusion_matrix(all_labels, all_preds))


VOICE RESULTS
Loss: 0.3239493492392846
Accuracy: 0.8395061728395061
ROC-AUC: 0.935672514619883

Classification Report:
              precision    recall  f1-score   support

           0       0.69      0.83      0.75        24
           1       0.92      0.84      0.88        57

    accuracy                           0.84        81
   macro avg       0.81      0.84      0.82        81
weighted avg       0.85      0.84      0.84        81


Confusion Matrix:
[[20  4]
 [ 9 48]]


Save predictions for fusion

In [40]:
def get_voice_predictions(model, meta_df, device):
    model.eval()
    rows = []

    with torch.no_grad():
        for _, row in meta_df.iterrows():
            file_path = row["file_path"]
            label = int(row["label"])
            attack_type = row.get("attack_type", None)

            try:
                feat = extract_logmel(file_path)
            except Exception:
                feat = np.zeros((64, 251), dtype=np.float32)

            feat = (feat - feat.mean()) / (feat.std() + 1e-6)
            feat = torch.tensor(feat).unsqueeze(0).unsqueeze(0).to(device)

            outputs = model(feat)
            prob_spoof = torch.softmax(outputs, dim=1)[:, 1].item()
            pred = 1 if prob_spoof >= 0.5 else 0

            rows.append({
                "file_path": file_path,
                "attack_type": attack_type,
                "true_label": label,
                "voice_pred": pred,
                "voice_prob_spoof": prob_spoof
            })

    return pd.DataFrame(rows)

voice_val_results = get_voice_predictions(model, val_df, device)

voice_acc = accuracy_score(voice_val_results["true_label"], voice_val_results["voice_pred"])
voice_auc = roc_auc_score(voice_val_results["true_label"], voice_val_results["voice_prob_spoof"])

print("\nVOICE VALIDATION RESULTS")
print("Accuracy:", voice_acc)
print("ROC-AUC:", voice_auc)
print("\nClassification Report:")
print(classification_report(voice_val_results["true_label"], voice_val_results["voice_pred"]))
print("\nConfusion Matrix:")
print(confusion_matrix(voice_val_results["true_label"], voice_val_results["voice_pred"]))

save_preds_path = "/content/drive/MyDrive/Final_Dataset/Sam_final_dataset/Audio/voice_val_predictions.csv"
voice_val_results.to_csv(save_preds_path, index=False)

print("Saved:", save_preds_path)
print("Saved model:", best_model_path)


VOICE VALIDATION RESULTS
Accuracy: 0.8395061728395061
ROC-AUC: 0.935672514619883

Classification Report:
              precision    recall  f1-score   support

           0       0.69      0.83      0.75        24
           1       0.92      0.84      0.88        57

    accuracy                           0.84        81
   macro avg       0.81      0.84      0.82        81
weighted avg       0.85      0.84      0.84        81


Confusion Matrix:
[[20  4]
 [ 9 48]]
Saved: /content/drive/MyDrive/Final_Dataset/Sam_final_dataset/Audio/voice_val_predictions.csv
Saved model: /content/drive/MyDrive/Final_Dataset/Sam_final_dataset/Audio/voice_binary_cnn_best.pth
